In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# Import the splitting tool
from sklearn.model_selection import train_test_split


##Data Cleanning

In [2]:
#read the file
df=pd.read_csv('/content/garments_worker_productivity.csv')
df

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1192,3/11/2015,Quarter2,finishing,Wednesday,10,0.75,2.90,NaN,960,0,0.0,0,0,8.0,0.628333
1193,3/11/2015,Quarter2,finishing,Wednesday,8,0.70,3.90,NaN,960,0,0.0,0,0,8.0,0.625625
1194,3/11/2015,Quarter2,finishing,Wednesday,7,0.65,3.90,NaN,960,0,0.0,0,0,8.0,0.625625
1195,3/11/2015,Quarter2,finishing,Wednesday,9,0.75,2.90,NaN,1800,0,0.0,0,0,15.0,0.505889


In [3]:
df.shape

(1197, 15)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   object 
 1   quarter                1197 non-null   object 
 2   department             1197 non-null   object 
 3   day                    1197 non-null   object 
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: f

In [5]:
#Number of Duplicate rows
df.duplicated().sum()

np.int64(0)

In [6]:
#Checking the missing values in each column before removing the duplicates
df.isnull().sum()

,0
date,0
quarter,0
department,0
day,0
team,0
targeted_productivity,0
smv,0
wip,506
over_time,0
incentive,0


In [7]:
# Check which departments have missing WIP
print(df.groupby('department')['wip'].apply(lambda x: x.isnull().sum()))

department
finishing     249
finishing     257
sweing          0
Name: wip, dtype: int64


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   object 
 1   quarter                1197 non-null   object 
 2   department             1197 non-null   object 
 3   day                    1197 non-null   object 
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: f

In [9]:
##Remove the space in finishing
# Remove the invisible spaces at the end of the words
df['department'] = df['department'].str.strip()

# Now check the unique values again
print(df['department'].unique())


['sweing' 'finishing']


In [10]:
#Fix Department Names (Typo & Spaces)
df['department'] = df['department'].str.strip().replace('sweing', 'sewing')

In [11]:
#Round up workers to nearest whole number
# Logic: You cannot have a fraction of a person for labor optimization
df['no_of_workers'] = np.ceil(df['no_of_workers']).astype(int)

In [12]:
#  Again Check which departments have missing WIP
print(df.groupby('department')['wip'].apply(lambda x: x.isnull().sum()))

department
finishing    506
sewing         0
Name: wip, dtype: int64


In [13]:
# Identify rows where 'no_of_workers' has a decimal part (not a whole number)
non_integer_workers = df[df['no_of_workers'] % 1 != 0]

# Count the number of such rows
num_rows = len(non_integer_workers)

print(f"Number of rows with decimal workers: {num_rows}")

# Display the unique decimal values found
print("Unique decimal values in the column:")
print(non_integer_workers['no_of_workers'].unique())

Number of rows with decimal workers: 0
Unique decimal values in the column:
[]


In [14]:
# Count the number of rows where actual_productivity is greater than 1
count = (df['actual_productivity'] > 1).sum()

print(f"Count of rows with productivity > 1: {count}")

Count of rows with productivity > 1: 37


In [15]:
#Cap Actual Productivity at 1.0
df['actual_productivity'] = df['actual_productivity'].clip(upper=1.0)


In [16]:
#Handle Quarter 5 (Optional: renaming to Quarter4 for consistency)
df['quarter'] = df['quarter'].replace('Quarter5', 'Quarter4')

In [17]:
#Convert Date to Datetime objects
df['date'] = pd.to_datetime(df['date'])
# Conver Int to object for team variable
df['team'] = df['team'].astype(object)
# Convert over time and incentive to the float
df[['over_time', 'incentive']] = df[['over_time', 'incentive']].astype(float)
# Conver _no_of_workers to the int
df['no_of_workers'] = df['no_of_workers'].astype(int)

In [18]:
# Dropping the team column from both sets before splitting
df.drop(columns=['team'], inplace=True, errors='ignore')

print("Team ID has been removed. The model will now focus purely on resource metrics.")

Team ID has been removed. The model will now focus purely on resource metrics.


In [19]:
df.drop(columns=['date'], inplace=True, errors='ignore')

##Split data set into trainnig and testing


In [20]:
# Perform a random split (80% train, 20% test)
# random_state ensures reproducibility
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

# Check number of rows in each part
print("train_data:", train_data.shape[0])
print("test_data:", test_data.shape[0])

train_data: 957
test_data: 240


In [21]:
train_data.isnull().sum()

,0
quarter,0
department,0
day,0
targeted_productivity,0
smv,0
wip,415
over_time,0
incentive,0
idle_time,0
idle_men,0


In [22]:

#Select numerical columns
num_cols = train_data.select_dtypes(include=['float64', 'int64']).columns

# Drop the specific column from the Index object
num_cols = num_cols.drop('actual_productivity')
train_numeric = train_data[num_cols]
test_numeric = test_data[num_cols]

In [23]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice = IterativeImputer(max_iter=10, random_state=42)

# Fit only on train
train_imputed = pd.DataFrame(
    mice.fit_transform(train_numeric),
    columns=num_cols,
    index=train_numeric.index
)

# Transform test (DO NOT FIT AGAIN)
test_imputed = pd.DataFrame(
    mice.transform(test_numeric),
    columns=num_cols,
    index=test_numeric.index
)

In [24]:
train_data['wip'] = train_imputed['wip']
test_data['wip'] = test_imputed['wip']

In [25]:
train_data

,quarter,department,day,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
1189,Quarter2,sewing,Wednesday,0.70,30.48,914.000000,6840.0,30.0,0.0,0,1,57,0.700505
575,Quarter1,finishing,Monday,0.75,3.94,1185.814897,2280.0,0.0,0.0,0,0,19,0.504596
76,Quarter1,finishing,Monday,0.75,2.90,1184.984148,960.0,0.0,0.0,0,0,8,0.763375
731,Quarter2,finishing,Thursday,0.70,4.15,1185.512804,1800.0,0.0,0.0,0,0,15,1.000000
138,Quarter2,sewing,Thursday,0.80,11.61,548.000000,15120.0,63.0,0.0,0,0,32,0.800107
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044,Quarter1,finishing,Tuesday,0.70,4.60,1186.494503,3360.0,0.0,0.0,0,0,8,0.354444
1095,Quarter1,finishing,Saturday,0.50,2.90,1184.984148,960.0,0.0,0.0,0,0,8,0.797500
1130,Quarter2,finishing,Monday,0.60,3.94,1184.573542,0.0,2880.0,0.0,0,0,12,0.864343
860,Quarter3,sewing,Thursday,0.75,30.10,444.000000,0.0,0.0,5.0,20,1,59,0.611141


outlier detection

In [27]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder



le = LabelEncoder()
cat_cols = ['quarter', 'department', 'day']
for col in cat_cols:
    train_data[col] = le.fit_transform(train_data[col])

# 2. Define the two scenarios
# Scenario A: Including the response variable (actual_productivity)
X_with = train_data.copy()

# Scenario B: Excluding the response variable (actual_productivity)
X_without = train_data.drop(columns=['actual_productivity'])

# 3. Detect outliers
iso = IsolationForest(contamination=0.05, random_state=42)

# Count outliers for 'With Response'
preds_with = iso.fit_predict(X_with)
count_with = list(preds_with).count(-1)

# Count outliers for 'Without Response'
preds_without = iso.fit_predict(X_without)
count_without = list(preds_without).count(-1)

print(f"Outliers detected (including response): {count_with}")
print(f"Outliers detected (excluding response): {count_without}")

Outliers detected (including response): 48
Outliers detected (excluding response): 48


In [31]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# --- Re-prepare the data for consistent encoding and outlier detection ---

# Assuming `df` is the DataFrame after all initial cleaning and MICE imputation
# but BEFORE any Label Encoding was applied to categorical columns.
# To ensure consistency, we will re-split and re-impute within this cell's context.

# Identify target variable
target_variable = 'actual_productivity'

# Separate features (X) and target (y)
X = df.drop(columns=[target_variable])
y = df[target_variable]

# Perform the train-test split again to get fresh copies with original data types
# This ensures train_data and test_data have their original string categorical columns
# and are not affected by previous inplace modifications.
X_train_repaired, X_test_repaired, y_train_repaired, y_test_repaired = train_test_split(X.copy(), y.copy(), test_size=0.2, random_state=42)

# Assign the feature sets to the variables used later for clarity
train_data_repaired = X_train_repaired
test_data_repaired = X_test_repaired

# MICE imputation step (re-applying as per original notebook flow for consistency)
# Select numerical columns (excluding the target if it's not part of MICE imputation)
num_cols_repaired = train_data_repaired.select_dtypes(include=['float64', 'int64']).columns

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice = IterativeImputer(max_iter=10, random_state=42)

# Fit only on repaired train numeric data
train_numeric_imputed = pd.DataFrame(
    mice.fit_transform(train_data_repaired[num_cols_repaired]),
    columns=num_cols_repaired,
    index=train_data_repaired.index
)

# Transform repaired test numeric data
test_numeric_imputed = pd.DataFrame(
    mice.transform(test_data_repaired[num_cols_repaired]),
    columns=num_cols_repaired,
    index=test_data_repaired.index
)

# Update the 'wip' column with imputed values for repaired datasets
train_data_repaired['wip'] = train_numeric_imputed['wip']
test_data_repaired['wip'] = test_numeric_imputed['wip']

# Combine features and target again for the IsolationForest
train_data_encoded = train_data_repaired.copy()
test_data_encoded = test_data_repaired.copy()

# --- Correct Label Encoding for Categorical Columns ---
cat_cols = ['quarter', 'department', 'day']
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    # Fit the LabelEncoder on the training data's string values
    train_data_encoded[col] = le.fit_transform(train_data_encoded[col])
    # Use the *fitted* encoder to transform the test data's string values
    # Handle potential unseen labels in test set by falling back to -1 or a unique new label
    # For simplicity, assuming all test categories are present in train, if not, an error might occur here
    # or a different encoder (e.g., OneHotEncoder or custom mapping) would be more robust.
    test_data_encoded[col] = le.transform(test_data_encoded[col])
    encoders[col] = le # Store encoders if needed for inverse_transform later

# Add the target variable back to the encoded train and test sets for outlier detection (if desired)
# The original problem statement showed X_with and X_without scenarios, meaning the target might be included.
# For IsolationForest, usually we fit on features only.
# Let's prepare X for IsolationForest

X_train = train_data_encoded # Features for IsolationForest
X_test = test_data_encoded   # Features for IsolationForest

# 2. Fit Isolation Forest ONLY on Training Data (using encoded data)
iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_train) # The model learns the 'normal' pattern here from the correctly encoded training data

# 3. Predict outliers on the Test Data (using encoded data)
# The model flags points in the test set that deviate from the 'normal' training pattern
test_preds = iso.predict(X_test)

# 4. View results
test_data_with_preds = X_test.copy() # Use the encoded test data for results
test_data_with_preds['outlier'] = test_preds

outliers_in_test = test_data_with_preds[test_data_with_preds['outlier'] == -1]
print(f"Number of outliers detected in test set: {len(outliers_in_test)}")


Number of outliers detected in test set: 9


In [33]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

# The data preparation and encoding steps are already correctly performed
# in cell 7iLGlqD4zWXj, which produced the `X_train` and `X_test` variables.
# We will use these pre-processed variables directly.

# 1. (Skip) Initialize dictionaries to store encoders for each categorical column
# 2. (Skip) Encode categorical columns consistently
# 3. Prepare datasets for Isolation Forest - use already prepared X_train and X_test

# `X_train` and `X_test` are already defined and correctly encoded/imputed
# from the successful execution of cell `7iLGlqD4zWXj`.
# They already exclude 'date' and are ready for model training.

# 4. Fit Isolation Forest on Training Data
iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_train) # Using the correctly prepared X_train from cell 7iLGlqD4zWXj

# 5. Predict Outliers
# Predict on test data
test_preds = iso.predict(X_test) # Using the correctly prepared X_test from cell 7iLGlqD4zWXj

# Add predictions to your test dataframe
# We need the original (but imputed/cleaned) test_data for context,
# but ensure it's the one corresponding to X_test (which is `test_data_encoded` from 7iLGlqD4zWXj).
# Since X_test itself is a copy of `test_data_encoded`, we can use X_test for this.
test_data_with_outliers = X_test.copy()
test_data_with_outliers['outlier'] = test_preds

# 6. View results
outliers_in_test = test_data_with_outliers[test_data_with_outliers['outlier'] == -1]

print(f"Number of outliers detected in test set: {len(outliers_in_test)}")
print(outliers_in_test.head())


Number of outliers detected in test set: 9
     quarter  department  day  targeted_productivity    smv      wip  \
761        1           1    1                   0.60  11.41   1039.0   
561        0           1    0                   0.80  22.94  16882.0   
979        3           1    1                   0.50  26.66   1448.0   
876        3           1    2                   0.75  11.41    834.0   
798        2           1    0                   0.70  30.10      7.0   

     over_time  incentive  idle_time  idle_men  no_of_style_change  \
761     2280.0       23.0        0.0         0                   2   
561     7020.0      113.0        0.0         0                   0   
979     6840.0       30.0        0.0         0                   2   
876     3480.0        0.0        0.0         0                   2   
798     7080.0       27.0        2.0        10                   2   

     no_of_workers  outlier  
761             55       -1  
561             59       -1  
979          